# TV1 - EDA và data processing

- Đếm dòng/series/dải ngày; schema, missing và duplicate keys.
- Phân tích sales=0, seasonality, promo, family/store.
- Kiểm tra tất cả bảng phụ; holiday locale/transfer; oil và transactions availability.
- Điền bảng trước/sau processing và lý do; không bịa kết quả.

## 1. Đọc dữ liệu

Mục tiêu:
- Đọc các bảng của cuộc thi Store Sales.
- Xác định kích thước và khoảng thời gian của dữ liệu.
- Chỉ khám phá lượng bán đến ngày 14/07/2017.
- Giữ riêng giai đoạn holdout cuối để đánh giá sau.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Tìm thư mục dữ liệu cuộc thi trên Kaggle.
candidates = [
    path.parent
    for path in Path("/kaggle/input").rglob("train.csv")
    if (path.parent / "stores.csv").exists()
    and (path.parent / "holidays_events.csv").exists()
]

assert len(candidates) == 1, (
    f"Cần đúng một nguồn dữ liệu cuộc thi, tìm thấy: {candidates}"
)

DATA = candidates[0]

# Thư mục lưu các kết quả EDA.
OUT = Path("/kaggle/working/eda")
OUT.mkdir(parents=True, exist_ok=True)

# Các bảng cần đọc.
file_names = [
    "train",
    "test",
    "stores",
    "holidays_events",
    "oil",
    "transactions",
    "sample_submission",
]

tables = {
    name: pd.read_csv(DATA / f"{name}.csv")
    for name in file_names
}

# Chuyển cột ngày từ chữ sang kiểu datetime.
for frame in tables.values():
    if "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"])

train_raw = tables["train"]
test = tables["test"]

# Chỉ dùng lịch sử đến mốc này để khám phá lượng bán.
EDA_CUTOFF = pd.Timestamp("2017-07-14")

train = train_raw.loc[
    train_raw["date"] <= EDA_CUTOFF
].copy()

print("Đường dẫn dữ liệu:", DATA)
print("Số dòng train gốc:", len(train_raw))
print("Số dòng train dùng cho EDA:", len(train))
print("Số dòng test:", len(test))
print("Ngày bắt đầu EDA:", train["date"].min())
print("Ngày kết thúc EDA:", train["date"].max())

display(train.head())

## 2. Tổng quan các bảng

Kiểm tra số dòng, số cột, dữ liệu thiếu, dòng trùng hoàn toàn
và khoảng thời gian của từng bảng.

In [ ]:
overview_rows = []

for name, frame in tables.items():
    overview_rows.append({
        "table": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "date_min": (
            str(frame["date"].min().date())
            if "date" in frame.columns else None
        ),
        "date_max": (
            str(frame["date"].max().date())
            if "date" in frame.columns else None
        ),
    })

overview = pd.DataFrame(overview_rows)

display(overview)

overview.to_csv(
    OUT / "table_overview.csv",
    index=False,
)

## 3. Kiểm tra chất lượng dữ liệu

Kiểm tra khóa trùng, giá trị thiếu theo cột, lượng bán và
khuyến mãi không hợp lệ, số cửa hàng và số nhóm sản phẩm.

In [ ]:
import numpy as np

KEY = ["date", "store_nbr", "family"]

checks = {
    "train_duplicate_key": int(train_raw.duplicated(KEY).sum()),
    "test_duplicate_key": int(test.duplicated(KEY).sum()),
    "train_duplicate_id": int(train_raw["id"].duplicated().sum()),
    "test_duplicate_id": int(test["id"].duplicated().sum()),

    "stores_duplicate_key": int(
        tables["stores"].duplicated(["store_nbr"]).sum()
    ),
    "transactions_duplicate_key": int(
        tables["transactions"].duplicated(
            ["date", "store_nbr"]
        ).sum()
    ),
    "oil_duplicate_date": int(
        tables["oil"].duplicated(["date"]).sum()
    ),

    "negative_sales": int((train_raw["sales"] < 0).sum()),
    "nonfinite_sales": int(
        (~np.isfinite(train_raw["sales"])).sum()
    ),
    "negative_promotion_train": int(
        (train_raw["onpromotion"] < 0).sum()
    ),
    "negative_promotion_test": int(
        (test["onpromotion"] < 0).sum()
    ),

    "n_stores": int(train_raw["store_nbr"].nunique()),
    "n_families": int(train_raw["family"].nunique()),
    "n_series": int(
        train_raw[["store_nbr", "family"]]
        .drop_duplicates()
        .shape[0]
    ),
    "n_test_dates": int(test["date"].nunique()),
}

print(json.dumps(checks, indent=2))

print("\nSố ô thiếu theo cột trong oil:")
display(tables["oil"].isna().sum().rename("missing"))

(OUT / "quality_checks.json").write_text(
    json.dumps(checks, indent=2),
    encoding="utf-8",
)

## 4. Kiểm tra ngày thiếu và độ phủ các chuỗi

Phân biệt:
- Có dòng dữ liệu và sales bằng 0.
- Không có dòng dữ liệu cho ngày đó.

Kiểm tra trên giai đoạn EDA đến ngày 14/07/2017.

In [ ]:
# Số tổ hợp cửa hàng–nhóm hàng quan sát được.
series_keys = train[["store_nbr", "family"]].drop_duplicates()
n_series = len(series_keys)

# Lịch đầy đủ trong khoảng EDA.
calendar = pd.date_range(
    train["date"].min(),
    train["date"].max(),
    freq="D",
)

# Đếm số dòng thực tế mỗi ngày.
daily_coverage = (
    train.groupby("date")
    .size()
    .reindex(calendar, fill_value=0)
    .rename("observed_rows")
    .to_frame()
)

daily_coverage.index.name = "date"
daily_coverage["expected_rows"] = n_series
daily_coverage["missing_rows"] = (
    daily_coverage["expected_rows"]
    - daily_coverage["observed_rows"]
)

# Ngày mất toàn bộ và ngày chỉ thiếu một phần.
fully_missing = daily_coverage.loc[
    daily_coverage["observed_rows"] == 0
]

partially_missing = daily_coverage.loc[
    (daily_coverage["observed_rows"] > 0)
    & (daily_coverage["missing_rows"] > 0)
]

print("Số ngày theo lịch:", len(calendar))
print("Số chuỗi:", n_series)
print("Số ngày không có dòng nào:", len(fully_missing))
display(fully_missing)

print("Số ngày thiếu một phần:", len(partially_missing))
display(partially_missing.head(20))

# Số ngày có quan sát của từng chuỗi.
series_coverage = (
    train.groupby(["store_nbr", "family"])
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observed_days=("date", "nunique"),
    )
    .reset_index()
)

series_coverage["calendar_days"] = len(calendar)
series_coverage["missing_days"] = (
    series_coverage["calendar_days"]
    - series_coverage["observed_days"]
)

print("Thống kê số ngày có/thiếu của các chuỗi:")
display(
    series_coverage[["observed_days", "missing_days"]]
    .describe()
)

daily_coverage.to_csv(OUT / "daily_coverage.csv")

series_coverage.to_csv(
    OUT / "series_coverage.csv",
    index=False,
)

### Nhận xét về độ phủ dữ liệu

Trong khoảng 01/01/2013–14/07/2017 có 1.656 ngày theo lịch.
Toàn bộ 1.782 chuỗi đều có 1.652 ngày quan sát và thiếu đúng
bốn ngày: 25/12/2013, 25/12/2014, 25/12/2015 và 25/12/2016.

Không phát hiện ngày chỉ thiếu một phần các chuỗi.

Các ngày thiếu trùng ngày Giáng sinh, nhưng chưa có đủ bằng
chứng xác định nguyên nhân. Giữ nguyên dữ liệu gốc và không
tự coi lượng bán của ngày thiếu bằng 0.

Khi tạo chuỗi đầu vào, cần giữ đúng lịch ngày và đánh dấu
quan sát bị thiếu để đặc trưng độ trễ không bị lệch ngày.

## 5. Phân phối lượng bán và tỷ lệ sales bằng 0

Chỉ phân tích các dòng thực sự có quan sát trong giai đoạn EDA.
Sales bằng 0 được phân biệt với những ngày không có dữ liệu.

In [ ]:
# Thống kê lượng bán và khuyến mãi trên phần dữ liệu EDA.
distribution = train[["sales", "onpromotion"]].describe(
    percentiles=[0.5, 0.9, 0.99]
)

display(distribution)

zero_count = int(train["sales"].eq(0).sum())
zero_ratio = float(train["sales"].eq(0).mean())

print("Số dòng sales = 0:", zero_count)
print(f"Tỷ lệ sales = 0: {zero_ratio:.2%}")

# Xem tỷ lệ bằng 0 theo từng nhóm sản phẩm.
zero_by_family = (
    train.assign(is_zero=train["sales"].eq(0))
    .groupby("family")
    .agg(
        observed_rows=("sales", "size"),
        zero_rows=("is_zero", "sum"),
        zero_ratio=("is_zero", "mean"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
    )
    .sort_values("zero_ratio", ascending=False)
)

display(zero_by_family)

distribution.to_csv(OUT / "sales_distribution.csv")

zero_by_family.to_csv(
    OUT / "zero_ratio_by_family.csv"
)

### Nhận xét về phân phối lượng bán

Trong giai đoạn EDA, có 930.772 dòng sales bằng 0, chiếm 31,62%
số dòng có quan sát. Các giá trị này được giữ nguyên và phân biệt
với những ngày không có dòng dữ liệu.

Phân phối sales lệch phải mạnh: trung bình khoảng 355,52, trong khi
trung vị chỉ bằng 10. Giá trị lớn nhất khoảng 124.717, cao hơn nhiều
so với phân vị 99% là 5.474.

Tỷ lệ sales bằng 0 và quy mô lượng bán khác nhau rõ rệt giữa các
nhóm sản phẩm. Vì vậy, cần giữ thông tin nhóm hàng và đánh giá
sai số theo từng nhóm.

Nhóm xem xét biến đổi log1p(sales) để giảm chênh lệch thang giá trị.
Không tự xóa sales bằng 0 hoặc cắt bỏ các giá trị lớn khi chưa
kiểm tra nguyên nhân. Hiệu quả của biến đổi cần được xác nhận
bằng thực nghiệm của các thành viên phụ trách mô hình.

## 6. Trực quan hóa phân phối lượng bán

So sánh sales gốc với log1p(sales).
Biến đổi log1p giữ sales bằng 0 ở giá trị 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(
    train["sales"],
    bins=100,
    color="steelblue",
)
axes[0].set_title("Phân phối sales gốc — toàn bộ phạm vi")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Số dòng")

axes[1].hist(
    np.log1p(train["sales"]),
    bins=100,
    color="darkorange",
)
axes[1].set_title("Phân phối log1p(sales)")
axes[1].set_xlabel("log1p(sales)")
axes[1].set_ylabel("Số dòng")

plt.tight_layout()
plt.savefig(
    OUT / "sales_raw_vs_log.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

(
    zero_by_family["zero_ratio"]
    .sort_values()
    .mul(100)
    .plot.barh(ax=ax, color="steelblue")
)

ax.set_title("Tỷ lệ sales bằng 0 theo nhóm sản phẩm")
ax.set_xlabel("Tỷ lệ (%)")
ax.set_ylabel("Nhóm sản phẩm")
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig(
    OUT / "zero_ratio_by_family.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

### Nhận xét trực quan

Phân phối sales gốc lệch phải mạnh. Biến đổi log1p nén các giá trị
lớn và giúp quan sát phần lượng bán dương rõ hơn, nhưng vẫn giữ
khối lượng lớn tại 0 và không tạo thành phân phối chuẩn.

Tỷ lệ sales bằng 0 khác biệt đáng kể giữa các nhóm sản phẩm.
BOOKS và BABY CARE có tỷ lệ cao nhất, trong khi nhiều nhóm hàng
khác có tỷ lệ thấp hơn rõ rệt.

Giữ nguyên các giá trị sales bằng 0. Không loại nhóm hàng hoặc
cắt bỏ giá trị lớn chỉ từ các biểu đồ này. Hiệu quả của biến đổi
log1p và mức sai số theo nhóm sẽ được kiểm chứng ở phần mô hình.

## 7. Xu hướng theo thời gian và chu kỳ tuần

Phân tích tổng lượng bán mỗi ngày và lượng bán trung bình theo thứ.
Các ngày thiếu dữ liệu được giữ là thiếu, không thay bằng sales = 0.

In [ ]:
# Tổng sales ở những ngày thực sự có quan sát.
daily_sales = (
    train.groupby("date")["sales"]
    .sum()
    .sort_index()
)

# Đưa về lịch đầy đủ; ngày thiếu vẫn là NaN.
daily_sales = daily_sales.reindex(
    pd.date_range(
        train["date"].min(),
        train["date"].max(),
        freq="D",
    )
)
daily_sales.index.name = "date"

# Chỉ tính trung bình khi có đủ 7 ngày quan sát liên tiếp.
rolling_7 = daily_sales.rolling(
    window=7,
    min_periods=7,
).mean()

fig, ax = plt.subplots(figsize=(14, 4))

daily_sales.plot(
    ax=ax,
    alpha=0.4,
    linewidth=0.7,
    label="Tổng sales mỗi ngày",
)

rolling_7.plot(
    ax=ax,
    linewidth=1.5,
    label="Trung bình trượt 7 ngày",
)

ax.set_title("Tổng lượng bán theo ngày — giai đoạn EDA")
ax.set_xlabel("Ngày")
ax.set_ylabel("Tổng sales")
ax.legend()

plt.tight_layout()
plt.savefig(
    OUT / "daily_sales_trend.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

daily_sales.rename("total_sales").to_csv(
    OUT / "daily_sales.csv"
)

In [ ]:
weekday_summary = (
    train.assign(weekday=train["date"].dt.dayofweek)
    .groupby("weekday")
    .agg(
        observed_rows=("sales", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
    )
    .reindex(range(7))
)

weekday_names = [
    "Thứ Hai",
    "Thứ Ba",
    "Thứ Tư",
    "Thứ Năm",
    "Thứ Sáu",
    "Thứ Bảy",
    "Chủ nhật",
]

display(weekday_summary)

fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(
    weekday_names,
    weekday_summary["mean_sales"],
    color="steelblue",
)

ax.set_title("Sales trung bình mỗi dòng theo thứ")
ax.set_xlabel("Thứ")
ax.set_ylabel("Mean sales")
ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(
    OUT / "weekday_sales.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

weekday_summary.to_csv(
    OUT / "weekday_summary.csv"
)

### Nhận xét về xu hướng và chu kỳ tuần

Tổng lượng bán theo ngày biến động mạnh nhưng đường trung bình
trượt 7 ngày cho thấy xu hướng tăng về dài hạn. Đây là xu hướng
tổng hợp của hệ thống, chưa xác định nguyên nhân tăng.

Sales trung bình mỗi dòng cao nhất vào Chủ nhật, khoảng 460,61,
tiếp theo là thứ Bảy, khoảng 431,36; thấp nhất vào thứ Năm,
khoảng 281,86.

Kết quả gợi ý sử dụng đặc trưng thứ trong tuần và lịch sử cùng
thứ. Hiệu quả dự báo cần được kiểm chứng trên validation.

Các đỉnh và đáy được giữ nguyên để điều tra thêm. Không tự
coi chúng là lỗi dữ liệu hoặc thay bằng giá trị trung bình.

## 8. Khuyến mãi và lượng bán

So sánh các dòng có và không có khuyến mãi trong giai đoạn EDA.
Đây là phân tích liên hệ, không phải kết luận về tác động nhân quả.

onpromotion là số mặt hàng đang được khuyến mãi, không phải
phần trăm giảm giá.

In [ ]:
# Tạo bảng phân tích riêng, không sửa dữ liệu train gốc.
promo_data = train.assign(
    has_promotion=train["onpromotion"].gt(0),
    is_zero=train["sales"].eq(0),
)

promo_summary = (
    promo_data.groupby("has_promotion")
    .agg(
        observed_rows=("sales", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        zero_ratio=("is_zero", "mean"),
    )
)

promo_summary["row_share"] = (
    promo_summary["observed_rows"] / len(promo_data)
)

display(promo_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels = ["Không khuyến mãi", "Có khuyến mãi"]
summary_plot = promo_summary.reindex([False, True])

axes[0].bar(
    labels,
    summary_plot["mean_sales"],
    color=["steelblue", "darkorange"],
)
axes[0].set_title("Sales trung bình")
axes[0].set_ylabel("Mean sales")

axes[1].bar(
    labels,
    summary_plot["median_sales"],
    color=["steelblue", "darkorange"],
)
axes[1].set_title("Sales trung vị")
axes[1].set_ylabel("Median sales")

plt.tight_layout()
plt.savefig(
    OUT / "promotion_sales_comparison.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

promo_summary.to_csv(
    OUT / "promotion_summary.csv"
)

# So sánh thêm trong từng nhóm hàng để tránh chỉ nhìn số gộp.
promo_by_family = (
    promo_data.groupby(["family", "has_promotion"])
    .agg(
        observed_rows=("sales", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
    )
    .reset_index()
)

display(promo_by_family.head(20))

promo_by_family.to_csv(
    OUT / "promotion_by_family.csv",
    index=False,
)

### Nhận xét về khuyến mãi

Các dòng có khuyến mãi chiếm khoảng 19,95% dữ liệu EDA, có sales
trung bình và trung vị cao hơn các dòng không có khuyến mãi.

Kết quả thể hiện mối liên hệ, chưa chứng minh khuyến mãi gây tăng
lượng bán. Khác biệt có thể liên quan đến nhóm hàng, cửa hàng,
thời điểm và việc lựa chọn sản phẩm để khuyến mãi.

Mức độ xuất hiện khuyến mãi khác nhau giữa các nhóm: BOOKS không
có dòng khuyến mãi trong giai đoạn EDA; BABY CARE chỉ có 52 dòng.

Giữ onpromotion làm đầu vào dự kiến. Hiệu quả dự báo sẽ được
kiểm chứng bằng thực nghiệm có và không có đặc trưng khuyến mãi.

## 9. Thông tin cửa hàng và kiểm tra phép ghép

Ghép metadata bằng store_nbr theo quan hệ nhiều-một.
Kiểm tra đủ metadata cho cả train và test, giữ nguyên số dòng
và thứ tự id.

In [ ]:
stores = tables["stores"].copy()

print("Kích thước bảng stores:", stores.shape)
display(stores.head())

print("Số giá trị khác nhau theo cột:")
display(stores.nunique().rename("unique_values"))

print("Số cửa hàng theo loại:")
display(stores.groupby("type").size().rename("n_stores"))

print("Số cửa hàng theo thành phố:")
display(
    stores.groupby("city")
    .size()
    .sort_values(ascending=False)
    .rename("n_stores")
)

assert stores["store_nbr"].notna().all()
assert stores["store_nbr"].is_unique

metadata_cols = [
    column
    for column in stores.columns
    if column != "store_nbr"
]

join_results = []

for name, frame in {"train": train, "test": test}.items():
    # Chỉ lấy khóa để kiểm tra phép ghép, giảm bộ nhớ.
    keys = frame[["id", "store_nbr"]]

    joined = keys.merge(
        stores,
        on="store_nbr",
        how="left",
        validate="many_to_one",
        indicator=True,
        sort=False,
    )

    unmatched_rows = int(
        joined["_merge"].ne("both").sum()
    )

    missing_metadata_rows = int(
        joined[metadata_cols].isna().any(axis=1).sum()
    )

    same_id_order = joined["id"].reset_index(drop=True).equals(
        keys["id"].reset_index(drop=True)
    )

    join_results.append({
        "dataset": name,
        "rows_before": len(keys),
        "rows_after": len(joined),
        "unmatched_rows": unmatched_rows,
        "missing_metadata_rows": missing_metadata_rows,
        "same_id_order": same_id_order,
    })

    assert len(joined) == len(keys), "Phép ghép làm đổi số dòng"
    assert unmatched_rows == 0, "Có cửa hàng không khớp"
    assert missing_metadata_rows == 0, "Metadata bị thiếu"
    assert same_id_order, "Thứ tự id bị thay đổi"

join_checks = pd.DataFrame(join_results)
display(join_checks)

stores.to_csv(OUT / "stores_metadata.csv", index=False)
join_checks.to_csv(OUT / "stores_join_checks.csv", index=False)

### Nhận xét về metadata cửa hàng

Bảng stores chứa 54 cửa hàng tại 22 thành phố và 16 tỉnh/bang,
gồm 5 loại cửa hàng và 17 cụm.

Phép ghép theo store_nbr trên tập EDA và test không làm thay đổi
số dòng hoặc thứ tự id. Không phát hiện cửa hàng không khớp hay
metadata bị thiếu.

Các cột store_nbr, city, state, type và cluster mang ý nghĩa
phân loại. Đặc biệt, cluster là mã cụm, không mặc định là đại lượng
có thứ tự hoặc khoảng cách số học.

Thông tin city và state sẽ được dùng để xác định phạm vi áp dụng
của ngày lễ địa phương.

## 10. Khảo sát ngày lễ và sự kiện

Kiểm tra loại sự kiện, phạm vi áp dụng, trạng thái chuyển ngày
và những ngày có nhiều sự kiện.

Nhiều dòng cùng ngày không mặc định là dữ liệu trùng.

In [ ]:
holidays = tables["holidays_events"].copy()

print("Kích thước bảng ngày lễ:", holidays.shape)
display(holidays.head(10))

print("Kiểu dữ liệu:")
display(holidays.dtypes.astype(str).rename("dtype"))

print("Số sự kiện theo loại:")
display(holidays["type"].value_counts().rename("n_events"))

print("Số sự kiện theo phạm vi:")
display(holidays["locale"].value_counts().rename("n_events"))

print("Các giá trị transferred:")
display(
    holidays["transferred"]
    .value_counts(dropna=False)
    .rename("n_events")
)

print("Bảng chéo loại sự kiện và transferred:")
display(
    pd.crosstab(
        holidays["type"],
        holidays["transferred"],
        dropna=False,
    )
)

# Một ngày có thể có nhiều sự kiện khác nhau.
events_per_date = holidays.groupby("date").size()

multi_event_dates = (
    events_per_date.loc[events_per_date > 1]
    .sort_values(ascending=False)
    .rename("n_events")
)

print("Số ngày có nhiều sự kiện:", len(multi_event_dates))
display(multi_event_dates.head(10))

# Kiểm tra tên địa phương có khớp metadata cửa hàng không.
local_names = set(
    holidays.loc[
        holidays["locale"].eq("Local"),
        "locale_name",
    ]
)

regional_names = set(
    holidays.loc[
        holidays["locale"].eq("Regional"),
        "locale_name",
    ]
)

print(
    "Tên Local không có trong city của stores:",
    sorted(local_names - set(stores["city"])),
)

print(
    "Tên Regional không có trong state của stores:",
    sorted(regional_names - set(stores["state"])),
)

# Xem ví dụ liên quan đến chuyển ngày.
transfer_examples = holidays.loc[
    holidays["transferred"].eq(True)
    | holidays["type"].eq("Transfer")
].sort_values(["date", "description"])

print("Ví dụ liên quan đến chuyển ngày:")
display(transfer_examples.head(20))

multi_event_dates.to_csv(
    OUT / "holiday_multi_event_dates.csv"
)

transfer_examples.to_csv(
    OUT / "holiday_transfer_examples.csv",
    index=False,
)

### Quyết định xử lý ngày lễ

Bảng có 350 sự kiện, trong đó 12 ngày lễ được đánh dấu chuyển
ngày và 12 sự kiện Transfer. Có 31 ngày chứa nhiều sự kiện.

Áp dụng National cho tất cả cửa hàng, Regional theo state,
Local theo city. Tất cả tên địa phương trong bảng sự kiện đều
khớp với metadata cửa hàng.

Holiday có transferred=True không được tính là ngày nghỉ thực tế
từ riêng sự kiện đó. Ngày Transfer được tính là ngày lễ hiệu lực.
Additional và Bridge cũng được đưa vào cờ ngày lễ hiệu lực.

Event và Work Day được giữ riêng. Gộp bằng max theo ngày–cửa hàng
để không nhân số dòng khi ghép vào dữ liệu sales.

Các cờ được tạo là ứng viên đầu vào. Event cần được kiểm tra
khả năng biết trước tại thời điểm dự báo, không mặc nhiên dùng
sự kiện tương lai làm đặc trưng.

## 11. Tạo đặc trưng ngày lễ theo ngày và cửa hàng

Tạo bảng duy nhất theo (date, store_nbr), giữ riêng từng loại cờ.

In [ ]:
# Chỉ lấy các cột cần thiết để tránh trùng tên "type".
store_locations = stores[["store_nbr", "city", "state"]]

national = holidays.loc[
    holidays["locale"].eq("National")
].merge(
    store_locations,
    how="cross",
)

regional = holidays.loc[
    holidays["locale"].eq("Regional")
].merge(
    store_locations,
    left_on="locale_name",
    right_on="state",
    how="inner",
)

local = holidays.loc[
    holidays["locale"].eq("Local")
].merge(
    store_locations,
    left_on="locale_name",
    right_on="city",
    how="inner",
)

# Mỗi dòng lúc này là một sự kiện áp dụng cho một cửa hàng.
expanded_events = pd.concat(
    [national, regional, local],
    ignore_index=True,
)

event_type = expanded_events["type"]
transferred = expanded_events["transferred"]

# Cờ ngày lễ hiệu lực.
expanded_events["is_holiday"] = (
    ~transferred
    & event_type.isin([
        "Holiday",
        "Transfer",
        "Additional",
        "Bridge",
    ])
).astype("uint8")

# Giữ các loại riêng để mô hình có thể phân biệt.
type_flags = {
    "is_transfer": "Transfer",
    "is_additional": "Additional",
    "is_bridge": "Bridge",
    "is_work_day": "Work Day",
    "is_event": "Event",
}

for flag, event_name in type_flags.items():
    expanded_events[flag] = (
        event_type.eq(event_name) & ~transferred
    ).astype("uint8")

expanded_events["is_transferred_original"] = (
    event_type.eq("Holiday") & transferred
).astype("uint8")

flag_cols = [
    "is_holiday",
    *type_flags.keys(),
    "is_transferred_original",
]

# Một dòng duy nhất cho mỗi ngày–cửa hàng.
event_flags = (
    expanded_events
    .groupby(["date", "store_nbr"], as_index=False)[flag_cols]
    .max()
)

# Lịch đủ ngày, bao gồm cả những ngày sales bị thiếu.
calendar_dates = pd.DataFrame({
    "date": pd.date_range(
        train_raw["date"].min(),
        test["date"].max(),
        freq="D",
    )
})

store_calendar = calendar_dates.merge(
    stores[["store_nbr"]],
    how="cross",
)

holiday_features = store_calendar.merge(
    event_flags,
    on=["date", "store_nbr"],
    how="left",
    validate="one_to_one",
)

# 0 ở đây nghĩa là không có sự kiện tương ứng trong lịch cung cấp.
# Không phải điền sales thiếu thành 0.
holiday_features[flag_cols] = (
    holiday_features[flag_cols]
    .fillna(0)
    .astype("uint8")
)

assert not holiday_features.duplicated(
    ["date", "store_nbr"]
).any()

assert len(holiday_features) == len(store_calendar)
assert not holiday_features[flag_cols].isna().any().any()

print("Kích thước bảng đặc trưng:", holiday_features.shape)
display(holiday_features.head())

holiday_features.to_csv(
    OUT / "holiday_features_by_store.csv",
    index=False,
)

In [ ]:
# Ngày lễ toàn quốc được chuyển sang 02/01/2017.
national_example = holiday_features.loc[
    holiday_features["date"].eq(pd.Timestamp("2017-01-02"))
]

print("Ngày 02/01/2017 — tổng cờ trên các cửa hàng:")
display(national_example[flag_cols].sum())

assert national_example["is_transfer"].eq(1).all()
assert national_example["is_holiday"].eq(1).all()

# Ngày lễ địa phương được chuyển tới 13/04/2017 tại Cuenca.
local_example = (
    holiday_features.loc[
        holiday_features["date"].eq(pd.Timestamp("2017-04-13"))
    ]
    .merge(
        store_locations,
        on="store_nbr",
        validate="one_to_one",
    )
)

print("Ngày 13/04/2017 — theo thành phố:")
display(
    local_example.groupby("city")[
        ["is_holiday", "is_transfer"]
    ].max()
)

# Kiểm tra ghép bảng cờ vào khóa dữ liệu.
holiday_join_results = []

for name, frame in {"train": train, "test": test}.items():
    keys = frame[["id", "date", "store_nbr"]]

    joined = keys.merge(
        holiday_features,
        on=["date", "store_nbr"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    same_id_order = joined["id"].reset_index(drop=True).equals(
        keys["id"].reset_index(drop=True)
    )

    missing_flags = int(
        joined[flag_cols].isna().any(axis=1).sum()
    )

    assert len(joined) == len(keys)
    assert same_id_order
    assert missing_flags == 0

    holiday_join_results.append({
        "dataset": name,
        "rows_before": len(keys),
        "rows_after": len(joined),
        "missing_flags": missing_flags,
        "same_id_order": same_id_order,
    })

holiday_join_checks = pd.DataFrame(holiday_join_results)
display(holiday_join_checks)

holiday_join_checks.to_csv(
    OUT / "holiday_join_checks.csv",
    index=False,
)

### Kiểm tra bảng đặc trưng ngày lễ

Bảng ngày lễ theo ngày–cửa hàng có 92.016 dòng và không trùng khóa.

Kiểm tra ngày 02/01/2017 cho thấy lễ chuyển ngày toàn quốc được
áp dụng cho đủ 54 cửa hàng. Ngày 13/04/2017 chỉ các cửa hàng
tại Cuenca nhận cờ lễ chuyển ngày trong ví dụ đã xét.

Phép ghép vào dữ liệu EDA và test giữ nguyên số dòng, thứ tự id
và không tạo giá trị thiếu ở các cờ.

Các đặc trưng là đầu vào ứng viên, chưa được tích hợp vào mô hình.
Cờ Event cần được kiểm tra khả năng biết trước tại thời điểm
dự báo trước khi sử dụng.

## 12. Kiểm tra dữ liệu giá dầu

Phân biệt giá bị thiếu trong các dòng có sẵn và ngày không xuất
hiện trong file. Kiểm tra lịch đầy đủ trước khi chọn cách xử lý.

Chỉ phân tích diễn biến giá đến mốc EDA. Không mặc định biết
giá dầu tương lai khi dự báo doanh số.

In [ ]:
oil = tables["oil"].sort_values("date").copy()

assert oil["date"].notna().all()
assert oil["date"].is_unique

print("Số dòng dầu gốc:", len(oil))
print("Khoảng ngày:", oil["date"].min(), "→", oil["date"].max())

# Phân biệt dòng có sẵn nhưng thiếu giá.
missing_price_rows = oil.loc[
    oil["dcoilwtico"].isna(),
    ["date", "dcoilwtico"],
].copy()

print("Số dòng có ngày nhưng thiếu giá:", len(missing_price_rows))
display(missing_price_rows.head(15))

# Chuyển về lịch đủ ngày trong phạm vi train/test.
full_calendar = pd.date_range(
    train_raw["date"].min(),
    test["date"].max(),
    freq="D",
)

oil_daily = (
    oil.set_index("date")[["dcoilwtico"]]
    .reindex(full_calendar)
)

oil_daily.index.name = "date"

oil_daily["date_in_original"] = (
    oil_daily.index.isin(oil["date"])
)

oil_daily["missing_price_original"] = (
    oil_daily["date_in_original"]
    & oil_daily["dcoilwtico"].isna()
)

oil_daily["missing_calendar_date"] = (
    ~oil_daily["date_in_original"]
)

oil_daily["weekday"] = oil_daily.index.dayofweek

coverage_summary = pd.Series({
    "calendar_days": len(oil_daily),
    "original_rows_in_range": int(
        oil_daily["date_in_original"].sum()
    ),
    "missing_price_in_existing_rows": int(
        oil_daily["missing_price_original"].sum()
    ),
    "dates_absent_from_file": int(
        oil_daily["missing_calendar_date"].sum()
    ),
    "total_missing_daily_price": int(
        oil_daily["dcoilwtico"].isna().sum()
    ),
})

display(coverage_summary.rename("count"))

print("Phân bố ngày thiếu theo thứ — 0 là Thứ Hai:")
display(
    oil_daily.groupby("weekday")[
        ["missing_price_original", "missing_calendar_date"]
    ].sum()
)

# Chỉ khảo sát giá trị trong phần lịch sử EDA.
oil_eda = oil_daily.loc[:EDA_CUTOFF].copy()

print("Thống kê giá dầu trong giai đoạn EDA:")
display(oil_eda["dcoilwtico"].describe())

print("Giá âm trong EDA:",
      int(oil_eda["dcoilwtico"].lt(0).sum()))

print("Các ngày đầu trong EDA:")
display(oil_eda.head(10))

coverage_summary.rename("count").to_csv(
    OUT / "oil_coverage_summary.csv"
)

oil_daily.to_csv(
    OUT / "oil_daily_quality.csv"
)

### Quyết định xử lý giá dầu

Trong khoảng lịch đầy đủ có 43 ngày có dòng nhưng thiếu giá và
486 ngày không xuất hiện trong file. Các ngày không xuất hiện
đều thuộc thứ Bảy và Chủ nhật.

Sử dụng forward fill để mang giá quan sát gần nhất về phía trước,
đồng thời giữ cờ thiếu giá ban đầu. Không dùng backward fill hoặc
nội suy dựa trên giá tương lai.

Ngày 01/01/2013 chưa có giá quá khứ nên giữ NaN. Không tự thay
giá dầu thiếu thành 0 hoặc loại các dòng sales tương ứng.

Khi dự báo nhiều ngày, chỉ sử dụng giá đã biết tại cutoff.
Giữ giá cuối đã biết là một phương án đầu vào ứng viên, cần
được đánh giá bằng thực nghiệm trước khi đưa vào mô hình.

## 13. Xử lý giá dầu bằng thông tin quá khứ

Giữ giá gốc, tạo giá forward fill và các cờ thiếu.

In [ ]:
oil_history = oil_eda[["dcoilwtico"]].copy()

oil_history["oil_missing_before_fill"] = (
    oil_history["dcoilwtico"].isna().astype("uint8")
)

oil_history["oil_price_ffill"] = (
    oil_history["dcoilwtico"].ffill()
)

oil_history["oil_still_missing"] = (
    oil_history["oil_price_ffill"].isna().astype("uint8")
)

print(
    "Ngày thiếu trước forward fill trong EDA:",
    int(oil_history["oil_missing_before_fill"].sum()),
)

print(
    "Ngày vẫn thiếu sau forward fill trong EDA:",
    int(oil_history["oil_still_missing"].sum()),
)

display(oil_history.head(10))

# Không điền ngược giá ngày 02/01 về ngày 01/01.
assert pd.isna(
    oil_history.loc["2013-01-01", "oil_price_ffill"]
)

# Cuối tuần dùng giá gần nhất trước đó.
assert (
    oil_history.loc["2013-01-05", "oil_price_ffill"]
    == oil_history.loc["2013-01-04", "dcoilwtico"]
)

# Những ngày đã có giá phải được giữ nguyên.
observed = oil_history["dcoilwtico"].notna()

pd.testing.assert_series_equal(
    oil_history.loc[observed, "dcoilwtico"],
    oil_history.loc[observed, "oil_price_ffill"],
    check_names=False,
)

oil_history.to_csv(
    OUT / "oil_history_eda.csv"
)

In [ ]:
cutoff = EDA_CUTOFF
forecast_end = cutoff + pd.Timedelta(days=16)

# Cắt dữ liệu TRƯỚC khi điền để không nhìn thấy giá tương lai.
oil_known = (
    oil.loc[oil["date"] <= cutoff]
    .set_index("date")["dcoilwtico"]
    .sort_index()
)

forecast_calendar = pd.date_range(
    train_raw["date"].min(),
    forecast_end,
    freq="D",
)

oil_at_origin = pd.DataFrame(index=forecast_calendar)
oil_at_origin.index.name = "date"

oil_at_origin["oil_price_available"] = (
    oil_known.reindex(forecast_calendar).ffill()
)

oil_at_origin["is_forecast_date"] = (
    oil_at_origin.index > cutoff
).astype("uint8")

future_oil = oil_at_origin.loc[
    oil_at_origin.index > cutoff
]

last_known_price = oil_known.dropna().iloc[-1]

assert len(future_oil) == 16
assert future_oil["oil_price_available"].eq(
    last_known_price
).all()

print("Cutoff:", cutoff.date())
print("Ngày có giá gần nhất:", oil_known.dropna().index[-1].date())
print("Giá cuối đã biết:", last_known_price)
display(future_oil)

future_oil.to_csv(
    OUT / "oil_forecast_example.csv"
)

### Kết quả xử lý giá dầu

Trong giai đoạn EDA có 515 ngày thiếu giá sau khi dựng lịch đủ.
Forward fill bổ sung được giá cho 514 ngày; ngày 01/01/2013
vẫn thiếu vì chưa có giá quá khứ.

Các ngày có quan sát giữ nguyên giá gốc. Giữ cờ thiếu ban đầu
để phân biệt giá quan sát với giá được điền.

Với cutoff 14/07/2017, giá cuối đã biết là 46,53. Ví dụ đầu vào
cho 16 ngày tiếp theo giữ cố định giá này, không sử dụng giá
thực tế sau cutoff.

Đây là phương án đặc trưng ứng viên, chưa được tích hợp hoặc
kiểm chứng hiệu quả trong mô hình.

## 14. Kiểm tra dữ liệu giao dịch

Kiểm tra khóa ngày–cửa hàng, giá trị giao dịch và độ phủ
so với các ngày–cửa hàng có dữ liệu sales trong giai đoạn EDA.

Không mặc định giao dịch thiếu bằng 0. Chỉ xem xét sử dụng
thông tin giao dịch đã biết tại thời điểm dự báo.

In [ ]:
transactions = tables["transactions"].copy()

transactions_eda = transactions.loc[
    transactions["date"] <= EDA_CUTOFF
].copy()

transaction_checks = {
    "raw_rows": len(transactions),
    "eda_rows": len(transactions_eda),
    "duplicate_date_store": int(
        transactions.duplicated(["date", "store_nbr"]).sum()
    ),
    "missing_cells": int(
        transactions.isna().sum().sum()
    ),
    "negative_transactions_eda": int(
        transactions_eda["transactions"].lt(0).sum()
    ),
    "zero_transactions_eda": int(
        transactions_eda["transactions"].eq(0).sum()
    ),
    "nonfinite_transactions_eda": int(
        (~np.isfinite(transactions_eda["transactions"])).sum()
    ),
    "unknown_stores": sorted(
        set(transactions["store_nbr"])
        - set(stores["store_nbr"])
    ),
}

print(json.dumps(transaction_checks, indent=2))

print(
    "Khoảng ngày giao dịch:",
    transactions["date"].min(),
    "→",
    transactions["date"].max(),
)

print("Thống kê giao dịch trong EDA:")
display(transactions_eda["transactions"].describe())

# Mỗi ngày–cửa hàng chỉ xuất hiện một lần để kiểm tra độ phủ.
sales_store_dates = (
    train[["date", "store_nbr"]]
    .drop_duplicates()
)

transaction_coverage = sales_store_dates.merge(
    transactions_eda[["date", "store_nbr", "transactions"]],
    on=["date", "store_nbr"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

transaction_coverage["missing_transaction_record"] = (
    transaction_coverage["_merge"].eq("left_only")
)

print("Số ngày–cửa hàng có dữ liệu sales:",
      len(transaction_coverage))

print("Số ngày–cửa hàng thiếu bản ghi giao dịch:",
      int(transaction_coverage[
          "missing_transaction_record"
      ].sum()))

print("Tỷ lệ thiếu bản ghi giao dịch:",
      f"{transaction_coverage['missing_transaction_record'].mean():.2%}")

coverage_by_store = (
    transaction_coverage.groupby("store_nbr")
    .agg(
        sales_store_days=("date", "size"),
        missing_transaction_days=(
            "missing_transaction_record", "sum"
        ),
    )
)

coverage_by_store["missing_ratio"] = (
    coverage_by_store["missing_transaction_days"]
    / coverage_by_store["sales_store_days"]
)

display(
    coverage_by_store.sort_values(
        "missing_ratio",
        ascending=False,
    ).head(15)
)

print("Ví dụ ngày–cửa hàng thiếu giao dịch:")
display(
    transaction_coverage.loc[
        transaction_coverage["missing_transaction_record"],
        ["date", "store_nbr"],
    ].head(15)
)

coverage_by_store.to_csv(
    OUT / "transactions_coverage_by_store.csv"
)

transaction_coverage.drop(columns="_merge").to_csv(
    OUT / "transactions_coverage_eda.csv",
    index=False,
)

(OUT / "transactions_quality_checks.json").write_text(
    json.dumps(transaction_checks, indent=2),
    encoding="utf-8",
);

### Nhận xét ban đầu về giao dịch

Trong giai đoạn EDA, có 7.448 ngày–cửa hàng có dòng sales nhưng
không có bản ghi transactions, chiếm khoảng 8,35%.

Mức thiếu không đồng đều giữa các cửa hàng. Cửa hàng 52 có tỷ lệ
thiếu cao nhất trong bảng kiểm tra, khoảng 94,79%.

Chưa xác định nguyên nhân thiếu. Không tự điền giao dịch bằng 0,
không forward fill qua các giai đoạn dài và không xóa sales
chỉ vì thiếu giao dịch.

Không sử dụng số giao dịch thật sau cutoff làm đầu vào dự báo.
Transactions chưa được chọn vào mô hình mặc định.

## 15. Đối chiếu giao dịch thiếu với lượng bán

Kiểm tra tổng lượng bán tại các ngày–cửa hàng thiếu transactions
và thời điểm xuất hiện bản ghi giao dịch đầu tiên.

Chỉ phân tích trong giai đoạn EDA.

In [ ]:
# Tổng sales của tất cả nhóm hàng trong mỗi ngày–cửa hàng.
store_daily_sales = (
    train.groupby(["date", "store_nbr"], as_index=False)
    .agg(total_sales=("sales", "sum"))
)

transaction_diagnostic = store_daily_sales.merge(
    transactions_eda[["date", "store_nbr", "transactions"]],
    on=["date", "store_nbr"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

transaction_diagnostic["missing_transaction_record"] = (
    transaction_diagnostic["_merge"].eq("left_only")
)

transaction_diagnostic["positive_sales"] = (
    transaction_diagnostic["total_sales"].gt(0)
)

print("Đối chiếu số ngày–cửa hàng:")
display(
    pd.crosstab(
        transaction_diagnostic["missing_transaction_record"],
        transaction_diagnostic["positive_sales"],
    )
)

# Chỉ xem những ngày thiếu bản ghi giao dịch.
missing_diagnostic = (
    transaction_diagnostic.loc[
        transaction_diagnostic["missing_transaction_record"]
    ]
    .groupby("store_nbr")
    .agg(
        missing_days=("date", "size"),
        missing_days_with_positive_sales=("positive_sales", "sum"),
    )
)

missing_diagnostic["missing_days_with_zero_sales"] = (
    missing_diagnostic["missing_days"]
    - missing_diagnostic["missing_days_with_positive_sales"]
)

display(
    missing_diagnostic.sort_values(
        "missing_days",
        ascending=False,
    ).head(15)
)

# Mốc đầu tiên CÓ QUAN SÁT, không mặc định là ngày khai trương.
first_sales_row = (
    train.groupby("store_nbr")["date"]
    .min()
    .rename("first_sales_row")
)

first_positive_sales = (
    store_daily_sales.loc[
        store_daily_sales["total_sales"].gt(0)
    ]
    .groupby("store_nbr")["date"]
    .min()
    .rename("first_positive_sales")
)

first_transaction = (
    transactions_eda.groupby("store_nbr")["date"]
    .min()
    .rename("first_transaction_record")
)

store_activity_dates = pd.concat(
    [
        first_sales_row,
        first_positive_sales,
        first_transaction,
    ],
    axis=1,
)

print("Ví dụ các cửa hàng có nhiều ngày thiếu:")
display(
    store_activity_dates.reindex([52, 22, 42, 21, 29, 20])
)

missing_diagnostic.to_csv(
    OUT / "transactions_missing_vs_sales.csv"
)

store_activity_dates.to_csv(
    OUT / "store_first_observed_dates.csv"
)

### Kết luận về transactions

Trong 7.448 ngày–cửa hàng thiếu bản ghi giao dịch:
- 7.330 ngày có tổng sales bằng 0.
- 118 ngày có tổng sales dương.

Ở các cửa hàng được kiểm tra minh họa, ngày sales dương đầu tiên
trùng với ngày có giao dịch đầu tiên. Đây là mốc quan sát trong
dữ liệu, không được xác định là ngày khai trương.

Không tự điền transactions thiếu bằng 0, không forward fill qua
các giai đoạn dài và không xóa các dòng sales tương ứng.

Transactions không nằm trong đầu vào mặc định. Nếu thử bổ sung,
chỉ sử dụng thông tin lịch sử đã biết tại forecast origin và
giữ rõ trạng thái thiếu dữ liệu. Không dùng giao dịch thật của
các ngày đang được dự báo.

## 16. Tổng hợp quyết định preprocessing

| Thành phần | Phát hiện | Quyết định |
|---|---|---|
| Khóa dữ liệu | Không trùng khóa ở các bảng đã kiểm tra | Kiểm tra khóa trước khi xử lý; không cần xóa trùng |
| Sales không hợp lệ | Không có sales âm hoặc NaN/inf | Giữ dữ liệu |
| Ngày sales thiếu | Thiếu 25/12 các năm 2013–2016 trên toàn bộ chuỗi | Giữ lịch đầy đủ và cờ quan sát; không tạo nhãn sales = 0 |
| Sales bằng 0 | 930.772 dòng, chiếm 31,62% trong EDA | Giữ nguyên, phân biệt với thiếu quan sát |
| Phân phối sales | Lệch phải, khác biệt lớn giữa nhóm hàng | Dùng log1p làm phương án biến đổi; chưa tự cắt ngoại lệ |
| Metadata cửa hàng | Ghép đủ, không tăng dòng hoặc đổi thứ tự id | Ghép many-to-one bằng store_nbr |
| Ngày lễ | Có chuyển ngày, phạm vi địa phương và nhiều sự kiện/ngày | Tạo cờ đúng phạm vi, gộp theo date–store_nbr trước khi ghép |
| Event | Có thể chứa sự kiện không biết trước | Chưa dùng mặc định; kiểm tra khả năng biết tại origin |
| Giá dầu | EDA thiếu 515 ngày, forward fill còn thiếu ngày đầu | Chỉ điền từ quá khứ, giữ cờ thiếu; không bfill |
| Dầu tương lai | Chưa biết tại origin theo giả định dự báo | Phương án ứng viên: giữ giá cuối đã biết trong 16 ngày |
| Transactions | 7.448 ngày–cửa hàng thiếu; 118 ngày có sales dương | Chưa dùng mặc định; không tự điền 0 |
| Chia dữ liệu | Bài toán dự báo theo thời gian | Validation theo thời gian; giữ riêng holdout cuối |

### Trạng thái triển khai

Các bước khảo sát và xử lý bảng phụ đã thực hiện trong notebook.
Chúng chưa tự động trở thành đầu vào của LightGBM hoặc LSTM.

Phần tiếp theo là chuyển các xử lý đã thống nhất thành hàm dùng
chung, bổ sung kiểm thử và bàn giao giao diện dữ liệu cho nhóm.

## 17. Đối chiếu hàm preprocessing với kết quả EDA

Đọc hàm từ source Dataset và so sánh với kết quả đã tạo trong
notebook trên cùng dữ liệu, khoảng ngày và cutoff.

Đây là kiểm tra tính nhất quán của triển khai, không phải
đánh giá hiệu quả dự báo.

In [ ]:
import importlib.util

# Tìm đúng file preprocessing trong Dataset source.
source_files = [
    path
    for path in Path("/kaggle/input").rglob("preprocessing.py")
    if path.parent.name == "store_sales"
    and (path.parents[2] / "pyproject.toml").exists()
]

assert len(source_files) == 1, (
    f"Cần đúng một preprocessing.py, tìm thấy: {source_files}"
)

# Đọc trực tiếp source để tránh dùng nhầm package đã cài.
spec = importlib.util.spec_from_file_location(
    "member1_preprocessing",
    source_files[0],
)

preprocessing = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocessing)

print("Source đang kiểm tra:", source_files[0])

# --------------------------------------------------
# 1. Đối chiếu đặc trưng ngày lễ.
# --------------------------------------------------
holiday_from_function = preprocessing.build_holiday_features(
    holidays=tables["holidays_events"],
    stores=tables["stores"],
    start=train_raw["date"].min(),
    end=test["date"].max(),
)

sort_keys = ["date", "store_nbr"]

expected_holiday = (
    holiday_features
    .sort_values(sort_keys)
    .reset_index(drop=True)
)

actual_holiday = (
    holiday_from_function[expected_holiday.columns]
    .sort_values(sort_keys)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    actual_holiday,
    expected_holiday,
    check_dtype=False,
)

print("PASS: Ngày lễ khớp kết quả EDA.")

# --------------------------------------------------
# 2. Đối chiếu giá dầu lịch sử và các cờ thiếu.
# --------------------------------------------------
oil_from_function = preprocessing.build_oil_features(
    oil=tables["oil"],
    start=train_raw["date"].min(),
    end=EDA_CUTOFF,
    cutoff=EDA_CUTOFF,
).set_index("date")

pd.testing.assert_series_equal(
    oil_from_function["oil_price"],
    oil_history["oil_price_ffill"],
    check_names=False,
    check_dtype=False,
    check_freq=False,
)

pd.testing.assert_series_equal(
    oil_from_function["oil_unobserved_at_origin"],
    oil_history["oil_missing_before_fill"],
    check_names=False,
    check_dtype=False,
    check_freq=False,
)

pd.testing.assert_series_equal(
    oil_from_function["oil_still_missing"],
    oil_history["oil_still_missing"],
    check_names=False,
    check_dtype=False,
    check_freq=False,
)

print("PASS: Giá dầu và cờ thiếu khớp kết quả EDA.")

# --------------------------------------------------
# 3. Đối chiếu giá dầu cho 16 ngày dự báo.
# --------------------------------------------------
oil_forecast_from_function = preprocessing.build_oil_features(
    oil=tables["oil"],
    start=EDA_CUTOFF + pd.Timedelta(days=1),
    end=EDA_CUTOFF + pd.Timedelta(days=16),
    cutoff=EDA_CUTOFF,
).set_index("date")

pd.testing.assert_series_equal(
    oil_forecast_from_function["oil_price"],
    future_oil["oil_price_available"],
    check_names=False,
    check_dtype=False,
    check_freq=False,
)

assert len(oil_forecast_from_function) == 16

assert (
    oil_forecast_from_function["oil_is_forecast"]
    .eq(1)
    .all()
)

print("PASS: Dầu dự báo khớp ví dụ EDA.")

# --------------------------------------------------
# 4. Lưu kết quả sau khi tất cả kiểm tra đều qua.
# --------------------------------------------------
verification = pd.DataFrame({
    "check": [
        "holiday_matches_eda",
        "oil_history_matches_eda",
        "oil_forecast_matches_eda",
    ],
    "status": ["PASS", "PASS", "PASS"],
})

verification.to_csv(
    OUT / "preprocessing_verification.csv",
    index=False,
)

display(verification)

In [ ]:
import shutil

result_zip = shutil.make_archive(
    "/kaggle/working/member1_eda_results",
    "zip",
    root_dir=OUT,
)

print("File kết quả:", result_zip)